In [31]:
# Script to train machine learning model.
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path
import joblib
# Add the necessary imports for the starter code.
from ml.data import process_data
from ml.model import *
# Add code to load in the data.
current_script_dir = Path.cwd()
data_path = current_script_dir.parent / "data" / "census.csv"

data = pd.read_csv(data_path)
data.columns = [col.strip() for col in data.columns] 
# Optional enhancement, use K-fold cross validation instead of a train-test split.
train,test = train_test_split(data, test_size=0.20, random_state=42)

cat_features = [
    "workclass",
    "education",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "native-country",
]

X_train, y_train, encoder, lb = process_data(
    train, categorical_features=cat_features, label="salary", training=True
)

# Proces the test data with the process_data function.
X_test, y_test, encoder, _ = process_data(
    test, categorical_features=cat_features, label="salary", training=False,encoder=encoder
)
# Train and save a model.
model = train_model(X_train, y_train)
joblib.dump(model, current_script_dir.parent / "model" / "random_forest_model.joblib")
joblib.dump(encoder, current_script_dir.parent / "model" / "encoder.joblib")

['/workspace/nd0821-c3-starter-code/starter/model/encoder.joblib']

In [32]:
X_train.shape

(26048, 108)

In [65]:
def slice_eval(
        data_cols,
        cat_features,
        label,
        x,
        y,
        slice_feature,
        model,
        lb,
        output_file_name = "slice_output.txt"
    ):
    output_lst = []
    
    num_features = [col for col in data_cols if col not in cat_features]
    num_features = [col for col in num_features if col != label]
    
    encoded_df = pd.DataFrame(
        x[:,len(num_features):],
        columns=encoder.get_feature_names_out(cat_features)
    )

    num_df = pd.DataFrame(
        x[:,:len(num_features)],
        columns = num_features
    )

    df = pd.concat([num_df,encoded_df],axis = 1)
    df[label] = y

    feature_cols = [col for col in df.columns if col.split("_")[0] == slice_feature]
    output_lst = [f"slice metrics using test data for the feature '{slice_feature}':\n\n"]

    for col in feature_cols:
        temp_df = df[df[col] == 1].reset_index(drop = True).copy()
        y = temp_df.pop(label)
        y_preds = model.predict(temp_df.values)

        # number of total rows
        total = y.shape[0]
        y_not_null = y[~y.isna()]
        
        # number of not null values
        not_null_num = y_not_null.shape[0]

        # remove null labels
        y_preds = y_preds[~y.isna()]
        y = y[~y.isna()]

        assert y_preds.shape[0] == y.shape[0]
        precision, recall, fbeta = compute_model_metrics(lb.transform(y),y_preds)
        output_lst.append(f"the metrics for the slice {col} are:\ntotal rows: {total} \nnot null rows: {not_null_num}\nprecision: {precision}\nrecall: {recall}\nfbeta: {fbeta}\n\n")
    
    with open("slice_output.txt", "w") as f:
        f.writelines(output_lst)

In [67]:
slice_eval(
        data_cols = data.columns,
        cat_features = cat_features,
        label = 'salary',
        x = X_test,
        y = y_test,
        slice_feature = 'education',
        model = model,
        lb = lb,
        output_file_name = "slice_output.txt"
    )

In [38]:
num_features = [col for col in data.columns if col not in cat_features]
num_features = [col for col in num_features if col != "salary"]
encoded_df = pd.DataFrame(
    X_test[:,len(num_features):],
    columns=encoder.get_feature_names_out(cat_features)
)

num_df = pd.DataFrame(
    X_test[:,:len(num_features)],
    columns = num_features
)

df = pd.concat([num_df,encoded_df],axis = 1)
df["salary"] = y_test

In [61]:
slice_feature = "education"
feature_cols = [col for col in df.columns if col.split("_")[0] == slice_feature]
output_lst = [f"slice metrics using test data for the feature '{slice_feature}':\n\n"]

In [62]:
for col in feature_cols:
    temp_df = df[df[col] == 1].reset_index(drop = True).copy()
    y = temp_df.pop("salary")
    y_preds = model.predict(temp_df.values)

    
    # number of total rows
    total = y.shape[0]
    y_not_null = y[~y.isna()]
    
    # number of not null values
    not_null_num = y_not_null.shape[0]

    # remove null labels
    y_preds = y_preds[~y.isna()]
    y = y[~y.isna()]

    assert y_preds.shape[0] == y.shape[0]
    precision, recall, fbeta = compute_model_metrics(lb.transform(y),y_preds)
    output_lst.append(f"the metrics for the slice {col} are:\ntotal rows: {total} \nnot null rows: {not_null_num}\nprecision: {precision}\nrecall: {recall}\nfbeta: {fbeta}\n\n")
    

In [63]:
with open("slice_output.txt", "w") as f:
    f.writelines(output_lst)

In [36]:


y_preds = model.predict(X_test)
precision, recall, fbeta = compute_model_metrics(lb.transform(y_test),y_preds)
print(f"the metrics for the slice {col} are:\nprecision: {precision}\nrecall: {recall}\nfbeta: {fbeta}\n")

the metrics for the slice sex_ Male are:
precision: 0.7418639053254438
recall: 0.6384468491406747
fbeta: 0.6862812179267875



(1.0, 1.0, 1.0)